# Early prediction of corporate insolvency from annual reports

DL and NLP mini project. Kartik Agrawal (I001), Ananya Baweja (I003)

We take the annual reports of 135 Indian listed companies that later went into insolvency (IBC) and
135 healthy companies of the same industry and size, and try to predict insolvency 1 to 3 years
before it happens from the report text and the financial ratios.

The notebook has five parts:

1. One report, start to finish: from the PDF to sections, language features and ratios
2. NLP on all 763 reports: language features, perplexity and FinBERT sentence vectors
3. Deep learning: baselines, the fusion network, the blend and the checks
4. Green AI: energy used by this run, and the greener options
5. Testing: upload any annual report and get a risk score

Part 1 uses the same code we used to build the dataset (`src/bpp` in our GitHub repo). Downloading
the 1,154 reports and processing all of them took several days on a laptop, so Parts 2 to 4 start
from the processed dataset.

To run: Runtime > Change runtime type > T4 GPU, then Runtime > Run all. It takes about an hour.
The `DL data` folder has to be somewhere in your Google Drive; the setup cell finds it.

## Setup

In [ ]:
# scikit-learn 1.9.1 is the version our saved models (Part 5) were trained with
!pip -q install "transformers>=4.40" codecarbon pypdfium2 "scikit-learn==1.9.1" 2>&1 | tail -1
!git clone -q https://github.com/ananya-baweja/bankruptcy-prediction.git /content/bpp-repo 2>/dev/null || git -C /content/bpp-repo pull -q

import sys, os, gzip, json, re, time, subprocess
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive
from scipy.stats import rankdata, ttest_rel, wilcoxon
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import sklearn
sys.path.insert(0, "/content/bpp-repo/src")      # our own code (bpp.nlp.lm for perplexity)

drive.mount("/content/drive")
# find the "DL data" folder: My Drive/DL data, My Drive/Colab/DL data, or any folder up to two levels down
DRIVE = Path("/content/drive/MyDrive")
found = [p.parent for pat in ("*/documents_labeled.csv", "*/*/documents_labeled.csv", "*/*/*/documents_labeled.csv")
         for p in DRIVE.glob(pat) if (p.parent/"folds.csv").exists()]
assert found, "Could not find the DL data folder (with documents_labeled.csv and folds.csv) in your Google Drive."
DATA  = found[0]
print("data folder:", DATA)
need = ["documents_labeled.csv", "folds.csv", "language_features.csv", "ratios.csv",
        "report_sections_1.jsonl.gz", "report_sections_2.jsonl.gz", "score/final_models/hgb.pkl"]
missing = [f for f in need if not (DATA/f).exists()]
assert not missing, f"Not in {DATA} yet (still uploading?): {missing}"
OUT   = DATA/"nlp_output";  OUT.mkdir(exist_ok=True)
CACHE = DATA/"_cache";      CACHE.mkdir(exist_ok=True)
dev   = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.is_available(), "| scikit-learn", sklearn.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then run again."

### Energy meter

We measure the energy of the heavy steps with codecarbon (India grid factor). We report the GPU
energy, because codecarbon's CPU number is only an estimate. If codecarbon fails, only the time is
recorded and the notebook carries on.

In [ ]:
import warnings, logging
GRID = "IND"            # India grid carbon factor (codecarbon's national average)

class EnergyMeter:
    """Measures one notebook step at a time: GREEN.start("name") ... GREEN.stop()."""
    def __init__(self): self.rows, self._t = [], None
    def start(self, name):
        self.name, self.t0, self._t = name, time.time(), None
        try:
            from codecarbon import OfflineEmissionsTracker
            logging.getLogger("codecarbon").setLevel(logging.ERROR)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                self._t = OfflineEmissionsTracker(country_iso_code=GRID, log_level="error",
                                                  save_to_file=False, measure_power_secs=1)
                self._t.start()
        except Exception as e:                       # the notebook never fails because of the meter
            print(f"  (energy meter unavailable: {e.__class__.__name__}; timing only)")
    def stop(self):
        secs = time.time() - self.t0
        row = dict(step=self.name, seconds=round(secs, 1), gpu_wh=None, cpu_wh_upper=None, g_co2e_gpu=None)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        if self._t is not None:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore"); kg = float(self._t.stop() or 0.0)
                d = self._t.final_emissions_data
                total_kwh, gpu_kwh = float(d.energy_consumed or 0.0), float(d.gpu_energy or 0.0)
                # CO2e of the GPU share only: the same grid factor (kg per kWh) applied to GPU energy
                row.update(gpu_wh=round(1000 * gpu_kwh, 3), cpu_wh_upper=round(1000 * float(d.cpu_energy or 0.0), 3),
                           g_co2e_gpu=round(1000 * kg * gpu_kwh / total_kwh, 3) if total_kwh else 0.0)
            except Exception as e:
                print(f"  (energy meter could not read codecarbon: {e.__class__.__name__}; timing only)")
        self.rows.append(row)
        print(f"  [energy] {self.name}: {secs:.0f}s, GPU {row['gpu_wh']} Wh, {row['g_co2e_gpu']} g CO2e (GPU)")
    def table(self): return pd.DataFrame(self.rows)

GREEN = EnergyMeter()
print("energy meter ready")

# Part 1: One report, start to finish

This is how every report in the dataset was processed, shown on one report: SEL Manufacturing's
annual report for FY2017 (the year before it was admitted to insolvency).

## 1.1 Read the PDF and find the sections

The text of each page is read from the PDF (pages that are scanned images go through OCR). Then our
rule-based section finder looks for the headings of the MD&A, the directors' report, the auditor's
report and the CARO annexure, skips the contents pages and cross-references, and cuts out each section.

In [ ]:
sys.path.insert(0, "/content/bpp-repo/final_run/scorer")
os.environ["BPP_LM_DICTIONARY"] = str(DATA/"score"/"loughran_mcdonald.csv")
import reader                                     # our PDF reader (same code that built the dataset)
cfg = reader.config()

DEMO_PDF = DATA/"score"/"reports"/"SEL_Manufacturing"/"FY2017.pdf"
doc = reader.read_pdf(DEMO_PDF, cfg)
print(f"{doc['company']}, FY{doc['fy']}: {doc['n_pages']} pages, {doc['n_ocr']} read by OCR\n")

print("sections found:")
for name, sec in doc["payload"]["sections"].items():
    if sec and sec.get("text"):
        where = f"pages {sec['start_page']}-{sec['end_page']}" if sec.get("start_page") else "inside the auditor's report"
        print(f"  {name:28s} {where:28s} {len(sec['text']):>7,} characters")

## 1.2 What the raw text looks like

The start of the MD&A as it comes out of the PDF: line breaks in the middle of sentences, headings
and numbering. The feature code works on this text.

In [ ]:
mdna_raw = doc["payload"]["sections"]["mdna"]["text"]
print(mdna_raw[:700])

## 1.3 From text to numbers

The language features of this one report (a few of them) and its 11 ratios from the financial
statements. The auditor's opinion is qualified (1) and the CARO annexure says the company has
defaulted on its loans: exactly the kind of signal the model learns from.

In [ ]:
lang = reader.language(doc)          # the engineered language features of this report
fin  = reader.financials(doc, cfg)   # statement figures and the 11 ratios

show = {
    "MD&A negative words (share)":      lang.get("mdna_lm_negative_ratio"),
    "MD&A uncertainty words (share)":   lang.get("mdna_lm_uncertainty_ratio"),
    "MD&A hedging density":             lang.get("mdna_hedge_density"),
    "MD&A Fog index":                   lang.get("mdna_fog_index"),
    "auditor negative words (share)":   lang.get("auditor_lm_negative_ratio"),
    "going-concern paragraph":          lang.get("has_going_concern"),
    "audit opinion (0 clean, 3 worst)": lang.get("audit_opinion_severity"),
    "CARO: loan default":               lang.get("caro_default_flag"),
    "CARO: unpaid statutory dues":      lang.get("caro_statutory_dues_flag"),
}
print(pd.Series(show).round(4).to_string())
print()
print(pd.Series({k: v for k, v in fin["ratios"].items() if k != "promoter_pledge"}).round(2).to_string())

# Part 2: NLP on all 763 reports

## 2.1 Load the report text

Each line of the two jsonl files is one annual report, already split into sections by our
extraction scripts. We use two sections: the MD&A (written by management) and the auditor's report.

In [ ]:
rows = []
for part in ["report_sections_1.jsonl.gz", "report_sections_2.jsonl.gz"]:
    with gzip.open(DATA/part, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            rows.append({"doc_id": r["doc_id"],
                         "mdna": r.get("mdna") or "",
                         "auditor_report": r.get("auditor_report") or ""})
text = pd.DataFrame(rows)
print(f"{len(text)} reports in the text export")
print(f"  with an MD&A section:            {int((text.mdna.str.len()>0).sum())}")
print(f"  with an auditor's report:        {int((text.auditor_report.str.len()>0).sum())}")
print(f"  median MD&A length:              {int(text.mdna.str.len().median()):,} characters")

## 2.2 Modelling table

Joins the labels, the language features and the ratios for each report.

The leakage rules were applied when the dataset was built: reports published within 180 days of the
insolvency admission, reports that discuss the insolvency petition, and reports filed under the
wrong year are not used. We do keep 44 healthy reports whose insolvent partner was removed, since a
healthy company's report can't leak anything. `strict` marks the 609 rows that follow the original
pair rule and have complete financials.

The next cell checks that the corrected CARO flags are being used.

In [ ]:
dl = pd.read_csv(DATA/"documents_labeled.csv")
lf = pd.read_csv(DATA/"language_features.csv")
ra = pd.read_csv(DATA/"ratios.csv")

RATIOS = ["current_ratio","quick_ratio","cash_to_assets","ebitda_margin","roce","roa",
          "debt_to_equity","debt_to_ebitda","interest_coverage","retained_earnings_to_assets",
          "altman_z_em"]

# keep everything the pipeline marked usable, plus the healthy partners described above
keep = dl.included | (dl.exclude_reason == "pair_partner_excluded")
base = dl.loc[keep & (dl.has_document == True),
              ["doc_id","firm_id","fy","pair_id","label","included","horizon"]].copy()

df = (base
      .merge(lf.drop(columns=[c for c in ["label","pair_id","role","horizon","included",
                                          "exclude_reason"] if c in lf.columns]),
             on=["doc_id","firm_id","fy"], how="left")
      .merge(ra[["firm_id","fy"]+RATIOS+["included_financials"]], on=["firm_id","fy"], how="left")
      .merge(text, on="doc_id", how="left")
      .fillna({"mdna":"", "auditor_report":""})
      .reset_index(drop=True))

# a flag so the model knows when the financial statements were incomplete, rather than
# silently receiving an imputed median as if it were real
df["financials_missing"] = (~df["included_financials"].fillna(False)).astype(float)
# the strict subset: rows that also satisfy the original 1:1 matched design
df["strict"] = (df.included & df.included_financials.fillna(False))

print(f"rows {len(df)}   healthy {int((df.label==0).sum())}   distressed {int((df.label==1).sum())}")
print(f"matched pairs {df.pair_id.nunique()}   strict subset {int(df.strict.sum())}")
print("\nexcluded, and why:")
print(dl.exclude_reason.value_counts(dropna=False).rename({np.nan:"(kept)"}).to_string())

In [ ]:
# The corrected CARO flags: before the fix 14% of healthy company-years carried a loan-default
# flag; after it about 6%. An old language_features.csv stops the notebook here.
rate = df.loc[df.label == 0, "caro_default_flag"].mean()
print(f"healthy company-years with a CARO loan-default flag: {rate:.1%}")
assert rate < 0.10, ("This is the language_features.csv from before the CARO correction. Copy "
                     "final_run/bpp_final/data_v2/language_features.csv into the data folder.")

## 2.3 Folds

5-fold cross-validation. The fold of every report is read from `folds.csv` so every run uses the
same split. A company and its matched peer are always in the same fold.

In [ ]:
N_SPLITS = 5
y, groups = df.label.to_numpy(), df.pair_id.to_numpy()
gkf = np.full(len(df), -1)
for k, (_, te) in enumerate(GroupKFold(n_splits=N_SPLITS).split(df, y, groups)):
    gkf[te] = k
fx = pd.read_csv(DATA/"folds.csv")[["doc_id","fold"]]
df = df.merge(fx, on="doc_id", how="left")
assert df.fold.notna().all() and len(df) == len(fx), "folds.csv does not match the table"
df["fold"] = df.fold.astype(int)
fold = df.fold.to_numpy(); y, groups = df.label.to_numpy(), df.pair_id.to_numpy()
print(f"this scikit-learn's GroupKFold agrees with folds.csv on {(gkf == fold).mean():.0%} of rows")

print(df.groupby("fold").agg(rows=("label","size"), distressed=("label","sum")).to_string())
assert all(len(set(groups[fold==k]) & set(groups[fold!=k])) == 0 for k in range(N_SPLITS)), \
    "a pair appears in two folds"
print("\ncheck passed: no matched pair is split across folds")

## 2.4 Engineered language features

These were computed by our code in `src/bpp/nlp`, separately for the MD&A and the auditor's report:

- tone: share of words from the Loughran-McDonald finance word lists (negative, positive,
  uncertainty, litigious, constraining, modal)
- hedging words and phrases ("may", "could", "subject to")
- an India-specific list of distress phrases ("one time settlement", "wilful defaulter", "SARFAESI")
- readability: Gunning Fog index, average sentence length, share of complex words
- the auditor's flags: going concern, emphasis of matter, type of opinion, and the two CARO flags
  (loan default, unpaid statutory dues)
- year-on-year change against the company's previous report

Here we pick the feature columns. We drop the ID columns, the two columns that count mentions of
insolvency (they would leak the answer) and any column that is empty or constant.

In [ ]:
DROP  = {"doc_id","firm_id","fy","pair_id","role","label","horizon","included","exclude_reason",
         "perplexity_source","drift_previous_doc_id","has_previous_year","strict","fold",
         "mdna","auditor_report","included_financials"}
LEAKY = {"cirp_specific_mentions","ibc_generic_mentions"}
LANG  = [c for c in lf.columns if c not in DROP and c not in LEAKY and lf[c].dtype.kind in "fib"]

# drop anything with no information on these rows (all-missing, or the same value everywhere)
dead = [c for c in LANG+RATIOS if df[c].isna().all() or df[c].nunique(dropna=True) <= 1]
LANG = [c for c in LANG if c not in dead] + ["financials_missing"]
print(f"{len(LANG)} language features, {len(RATIOS)} ratios")
print("dropped (no information):", dead)

Checking one feature by hand: the Fog index is 0.4 x (words per sentence + % of words with three or
more syllables). Here we work it out step by step for a few reports, using the same word, sentence
and syllable rules as our feature code, and compare it with the stored value.

In [ ]:
from bpp.nlp.lexicon import sentences, words
from bpp.nlp.readability import is_complex

def fog_by_hand(text):
    toks  = words(text)                      # lower-case alphabetic words
    sents = sentences(text)                  # same sentence splitter as the feature code
    avg_len = len(toks) / len(sents)
    complex_pct = 100 * sum(is_complex(t) for t in toks) / len(toks)   # 3+ syllables
    return avg_len, complex_pct, 0.4 * (avg_len + complex_pct)

sample = df[df.mdna.str.len() > 2000].head(5)
print(f"{'report':18s} {'words/sentence':>15s} {'% complex':>10s} {'Fog':>6s} {'stored':>7s}")
for _, r in sample.iterrows():
    a, c, f = fog_by_hand(r.mdna)
    print(f"{r.doc_id:18s} {a:15.1f} {c:10.1f} {f:6.1f} {r.mdna_fog_index:7.1f}")

## 2.5 Perplexity

A trigram language model (Kneser-Ney smoothing, our own code in `src/bpp/nlp/lm.py`) is trained on
the MD&A of the healthy companies in the training folds, and then scores each test report. A high
perplexity would mean the report doesn't read like a normal company's. It is fitted again for every
fold, so the test reports are never part of training.

In [ ]:
from bpp.nlp.lm import fit as lm_fit, perplexity as lm_perplexity

GREEN.start("perplexity language models (CPU)")
ppl = np.full(len(df), np.nan); t0 = time.time()
for k in range(N_SPLITS):
    train_healthy = (df.fold != k) & (df.label == 0) & (df.mdna.str.len() > 0)
    model = lm_fit(df.loc[train_healthy, "mdna"].tolist(), order=3, min_count=2)
    for i in np.flatnonzero((df.fold == k).to_numpy()):
        ppl[i] = lm_perplexity(model, df.mdna.iloc[i]) or np.nan
    print(f"  fold {k}: fitted on {int(train_healthy.sum())} healthy reports, "
          f"vocabulary {len(model.vocabulary):,}")
GREEN.stop()
df["perplexity_fold"] = ppl
LANG = LANG + ["perplexity_fold"]

ok = df.perplexity_fold.notna()
print(f"\ndone in {time.time()-t0:.0f}s")
print(f"median perplexity - distressed {df.loc[ok & (df.label==1),'perplexity_fold'].median():.1f}"
      f"  |  healthy {df.loc[ok & (df.label==0),'perplexity_fold'].median():.1f}")

In our final run this feature didn't work (ROC-AUC about 0.50). Indian MD&As follow a very standard
template, so they all look normal to a trigram model. We kept it and report it as a negative result.

## 2.6 FinBERT sentence vectors

Each section is split into sentences (sentences under 5 words are dropped). Each sentence goes
through FinBERT, which we keep frozen, and we average the token vectors to get 768 numbers per
sentence.

Only the first 96 sentences of each section are embedded, because the network in Part 3 reads 96.
The arrays are still 200 slots long, so Part 3 doesn't need to change. FinBERT runs in fp16 and the
vectors are cached in `_cache/`, so running again takes seconds.

In [ ]:
from transformers import AutoTokenizer, AutoModel
MODEL_NAME, MAX_SENTS, MAX_TOKENS, DIM = "ProsusAI/finbert", 200, 96, 768
READ_SENTS = 96   # Green AI: Part 2's network reads 96 sentence slots, so only these are embedded

_SPLIT, _WS = re.compile(r"(?<=[.!?])\s+"), re.compile(r"\s+")
def to_sentences(t, m=MAX_SENTS):
    """Split into sentences, keep those with 5+ words (drops page numbers and table debris)."""
    out = []
    for s in _SPLIT.split(t or ""):
        s = _WS.sub(" ", s).strip()
        if len(s.split()) >= 5:
            out.append(s)
            if len(out) >= m: break
    return out

def embed(section):
    tag = f"{section}_{MODEL_NAME.split('/')[-1]}_{MAX_SENTS}x{MAX_TOKENS}"
    npy, idx = CACHE/f"embstore_{tag}.npy", CACHE/f"embstore_{tag}.json"
    have, store = {}, np.zeros((0, MAX_SENTS, DIM), np.float16)
    if npy.exists() and idx.exists():
        store = np.load(npy); have = {d:i for i,d in enumerate(json.loads(idx.read_text()))}
    sents = df[section].map(to_sentences)
    need  = [d for d in df.doc_id if d not in have]
    print(f"[{section}] {len(have)} cached, {len(need)} to embed")
    if need:
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        enc = AutoModel.from_pretrained(MODEL_NAME).to(dev).eval().half()
        sent_of = dict(zip(df.doc_id, sents))
        new = np.zeros((len(need), MAX_SENTS, DIM), np.float16); t0 = time.time()
        with torch.no_grad():
            for j, d in enumerate(need):
                ss = sent_of[d][:READ_SENTS]                          # Green AI: skip slots never read
                for a in range(0, len(ss), 64):                       # sub-batch within a report
                    chunk = ss[a:a+64]
                    b = tok(chunk, padding=True, truncation=True,
                            max_length=MAX_TOKENS, return_tensors="pt").to(dev)
                    h  = enc(**b).last_hidden_state                   # (n, tokens, 768)
                    mk = b["attention_mask"].unsqueeze(-1).half()
                    new[j, a:a+len(chunk)] = ((h*mk).sum(1)/mk.sum(1).clamp(min=1)).cpu().numpy()
                if j % 100 == 0: print(f"   {j}/{len(need)}  {time.time()-t0:.0f}s", flush=True)
        store = np.concatenate([store, new], 0); order = list(have.keys()) + need
        np.save(npy, store); idx.write_text(json.dumps(order))
        have = {d:i for i,d in enumerate(order)}
        del enc; torch.cuda.empty_cache()
    E = store[[have[d] for d in df.doc_id]]          # reorder to match the table, row for row
    lengths = sents.map(len).to_numpy()
    return E, lengths, sents

GREEN.start("FinBERT embeddings, fp16 (GPU)")
E_mdna, L_mdna, S_mdna = embed("mdna")
E_aud,  L_aud,  S_aud  = embed("auditor_report")
GREEN.stop()
print(f"\nMD&A embeddings     {E_mdna.shape}   median {int(np.median(L_mdna))} sentences/report")
print(f"auditor embeddings  {E_aud.shape}   median {int(np.median(L_aud))} sentences/report")
skipped = int(np.clip(L_mdna - READ_SENTS, 0, None).sum() + np.clip(L_aud - READ_SENTS, 0, None).sum())
print(f"sentences not embedded because the network never reads them: {skipped:,} "
      f"of {int(L_mdna.sum() + L_aud.sum()):,}")

## 2.7 Save for Part 3

Part 3 only uses these files.

In [ ]:
df["sent_len_mdna"]    = L_mdna
df["sent_len_auditor"] = L_aud

TABLE_COLS = (["doc_id","firm_id","fy","pair_id","label","horizon","included","strict","fold",
               "sent_len_mdna","sent_len_auditor"] + RATIOS + LANG)
df[TABLE_COLS].to_csv(OUT/"modelling_table.csv", index=False)
np.save(OUT/"embeddings_mdna.npy",    E_mdna)
np.save(OUT/"embeddings_auditor.npy", E_aud)
(OUT/"feature_columns.json").write_text(json.dumps(
    {"ratios": RATIOS, "language": LANG, "max_sents": MAX_SENTS, "embedded_sents": READ_SENTS, "dim": DIM,
     "n_splits": N_SPLITS, "doc_id_order": df.doc_id.tolist()}, indent=2))

GREEN.table().to_csv(OUT/"energy_log.csv", index=False)
print("written to", OUT)
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:28s} {f.stat().st_size/1e6:8.1f} MB")
print(f"\n{len(df)} rows, {len(RATIOS)} ratios, {len(LANG)} language features")
print("Row order in the CSV matches row order in both .npy files. Part 2 relies on this.")

# Part 3: Deep learning

## 3.1 Load

Reads what Part 2 saved and makes a mask of which sentence slots hold real sentences.

In [ ]:
df   = pd.read_csv(OUT/"modelling_table.csv")
meta = json.loads((OUT/"feature_columns.json").read_text())
RATIOS, LANG = meta["ratios"], meta["language"]
MAX_SENTS, DIM, N_SPLITS = meta["max_sents"], meta["dim"], meta["n_splits"]
E_mdna = np.load(OUT/"embeddings_mdna.npy")
E_aud  = np.load(OUT/"embeddings_auditor.npy")

assert df.doc_id.tolist() == meta["doc_id_order"], "table and embeddings are out of step"
assert len(df) == len(E_mdna) == len(E_aud)

y      = df.label.to_numpy().astype(np.float32)
groups = df.pair_id.to_numpy()
FOLD   = df.fold.to_numpy()
STRICT = df.strict.to_numpy().astype(bool)
# a mask marking which sentence slots are real text rather than padding
M_mdna = np.arange(MAX_SENTS)[None,:] < df.sent_len_mdna.to_numpy()[:,None]
M_aud  = np.arange(MAX_SENTS)[None,:] < df.sent_len_auditor.to_numpy()[:,None]

print(f"{len(df)} rows | {int((y==1).sum())} distressed | {df.pair_id.nunique()} pairs")
print(f"{len(RATIOS)} ratios, {len(LANG)} language features, embeddings {E_mdna.shape}")

## 3.2 How we measure

PR-AUC is the main metric and ROC-AUC is shown next to it. Scores are computed in each fold and then
averaged, because every fold has its own model. `rank_norm` turns scores into ranks inside a fold, so
the scores of two models can be averaged.

In [ ]:
def rank_norm(p):
    """Map each fold's predictions onto (0,1) ranks. Ordering within a fold is untouched."""
    out = np.full(len(p), np.nan)
    for k in range(N_SPLITS):
        m = (FOLD == k) & ~np.isnan(p)
        if m.sum(): out[m] = rankdata(p[m]) / (m.sum() + 1)
    return out

def per_fold(p, subset=None):
    """ROC-AUC and PR-AUC in each fold, optionally restricted to a subset of rows."""
    roc, pr = [], []
    for k in range(N_SPLITS):
        m = (FOLD == k) & ~np.isnan(p)
        if subset is not None: m = m & subset
        if m.sum() < 5 or len(set(y[m])) < 2: continue
        roc.append(roc_auc_score(y[m], p[m])); pr.append(average_precision_score(y[m], p[m]))
    return np.array(roc), np.array(pr)

def report(p, name, subset=None):
    roc, pr = per_fold(p, subset)
    print(f"  {name:32s} ROC {roc.mean():.3f}+/-{roc.std(ddof=1):.3f}   "
          f"PR {pr.mean():.3f}+/-{pr.std(ddof=1):.3f}")
    return pr

## 3.3 Baselines

Gradient boosting on the ratios only, and on the ratios plus the language features. The difference
between the two is what the NLP features add.

In [ ]:
Xr = df[RATIOS].apply(pd.to_numeric, errors="coerce").to_numpy(float)
Xb = df[RATIOS+LANG].apply(pd.to_numeric, errors="coerce").to_numpy(float)

GREEN.start("gradient boosting, both baselines (CPU)")
oof_ratio, oof_both = np.zeros(len(df)), np.zeros(len(df))
for k in range(N_SPLITS):
    tr, te = FOLD != k, FOLD == k
    oof_ratio[te] = HistGradientBoostingClassifier(random_state=0).fit(Xr[tr], y[tr]).predict_proba(Xr[te])[:,1]
    oof_both[te]  = HistGradientBoostingClassifier(random_state=0).fit(Xb[tr], y[tr]).predict_proba(Xb[te])[:,1]

GREEN.stop()
print("baselines (all rows):")
report(oof_ratio, "ratios only"); report(oof_both, "ratios + language")

## 3.4 The fusion network

Three branches:

- text: a bidirectional GRU reads the 96 sentence vectors, and attention combines them into one vector
- language features: a small two-layer network
- ratios: a small two-layer network with batch norm

A softmax gate learns how much weight to give each branch for each report, and the head turns the
result into a risk score. These sizes came from a hyperparameter search we did earlier; with about
490 training reports per fold a small network works better.

In [ ]:
class AdditiveAttention(nn.Module):
    """Learns a weight for every sentence, then returns their weighted sum."""
    def __init__(self, d, hidden):
        super().__init__()
        self.proj, self.v = nn.Linear(d, hidden), nn.Linear(hidden, 1, bias=False)
    def forward(self, h, mask):
        score = self.v(torch.tanh(self.proj(h))).squeeze(-1)
        score = score.masked_fill(~mask, torch.finfo(score.dtype).min)   # padding gets no weight
        w = torch.softmax(score, 1).unsqueeze(-1)
        return (h * w).sum(1), w.squeeze(-1)

def mlp(i, h, o, drop, bn=False):
    layers = [nn.Linear(i, h)]
    if bn: layers.append(nn.BatchNorm1d(h))
    return nn.Sequential(*(layers + [nn.ReLU(), nn.Dropout(drop), nn.Linear(h, o), nn.ReLU()]))

class FusionNet(nn.Module):
    def __init__(self, n_lang, n_ratio, gru=32, block=64, attn_hidden=32, drop=0.5, dim=768):
        super().__init__()
        self.gru  = nn.GRU(dim, gru, batch_first=True, bidirectional=True)
        self.attn = AdditiveAttention(2*gru, attn_hidden)
        self.text_proj = nn.Sequential(nn.Linear(2*gru, block), nn.ReLU())
        self.lang  = mlp(n_lang,  block, block, drop)
        self.ratio = mlp(n_ratio, block, block, drop, bn=True)
        self.gate  = nn.Linear(3*block, 3)
        self.head  = nn.Sequential(nn.Linear(3*block, block), nn.ReLU(),
                                   nn.Dropout(drop), nn.Linear(block, 1))

    def forward(self, emb, mask, lang, ratio):
        # read exactly the real sentences: packing skips the zero padding, which the backward
        # direction of an unpacked GRU would otherwise run through
        lengths = mask.sum(1)
        packed = pack_padded_sequence(emb, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.gru(packed)
        h, _ = pad_packed_sequence(out, batch_first=True, total_length=mask.shape[1])
        pooled, attn_w = self.attn(h, mask)
        # a report with no usable text contributes an all-zero text block
        t = self.text_proj(pooled) * mask.any(1, keepdim=True).float()
        l, r = self.lang(lang), self.ratio(ratio)
        g = torch.softmax(self.gate(torch.cat([t, l, r], -1)), -1).unsqueeze(-1)   # (B,3,1)
        fused = (torch.stack([t, l, r], 1) * g).flatten(1)                          # (B,192)
        return self.head(fused).squeeze(-1), g.squeeze(-1), attn_w

CFG = dict(gru=32, block=64, attn_hidden=32, drop=0.50, lr=1e-3, wd=0.0,
           batch=16, gamma=1.0, n_sents=96, epochs=40, patience=8)
n_par = sum(p.numel() for p in FusionNet(len(LANG), len(RATIOS)).parameters() if p.requires_grad)
print(f"trainable parameters: {n_par:,}   (FinBERT's ~110,000,000 stay frozen)")

## 3.5 Training

Focal loss (focuses on the harder examples) with a little normal cross-entropy, AdamW, and early
stopping on an inner split of the training folds that is also grouped by pair. Missing values are
filled and features are scaled using the training fold only.

In [ ]:
def focal_bce(logit, target, gamma=1.0, ce_weight=0.1):
    ce = F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    if gamma == 0: return ce.mean()
    return ((1 - torch.exp(-ce))**gamma * ce).mean() + ce_weight * ce.mean()

def prep(train_idx, idx, X):
    """Fill gaps with the TRAINING median and scale by TRAINING mean/sd."""
    med = np.nanmedian(X[train_idx], 0); med = np.where(np.isnan(med), 0, med)
    mu  = np.nanmean(X[train_idx], 0);   sd = np.nanstd(X[train_idx], 0)
    mu  = np.where(np.isnan(mu), 0, mu)
    sd  = np.where(~np.isfinite(sd) | (sd < 1e-6), 1.0, sd)
    return ((np.where(np.isnan(X[idx]), med, X[idx]) - mu) / sd).astype(np.float32)

Xl_all = df[LANG].apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)
Xr_all = df[RATIOS].apply(pd.to_numeric, errors="coerce").to_numpy(np.float32)

def train_fold(k, E, MASK, labels=None, cfg=CFG, seed=0, keep_model=False):
    yy = y if labels is None else labels
    te = np.flatnonzero(FOLD == k); dv = np.flatnonzero(FOLD != k)
    a, b = next(GroupShuffleSplit(1, test_size=0.2, random_state=seed)
                .split(dv, yy[dv], groups[dv]))
    tr, va = dv[a], dv[b]                      # inner split, also grouped by pair
    torch.manual_seed(seed); np.random.seed(seed)
    ns = cfg["n_sents"]

    def dataset(idx):
        # only the first ns sentence slots go in; the mask marks the real sentences among them
        return TensorDataset(torch.from_numpy(np.ascontiguousarray(E[idx, :ns]).astype(np.float32)),
                             torch.from_numpy(np.ascontiguousarray(MASK[idx, :ns])),
                             torch.from_numpy(prep(tr, idx, Xl_all)),
                             torch.from_numpy(prep(tr, idx, Xr_all)),
                             torch.from_numpy(yy[idx]))
    Ltr = DataLoader(dataset(tr), batch_size=cfg["batch"], shuffle=True, drop_last=True)
    Lva = DataLoader(dataset(va), batch_size=128)
    Lte = DataLoader(dataset(te), batch_size=128)

    model = FusionNet(Xl_all.shape[1], Xr_all.shape[1],
                      cfg["gru"], cfg["block"], cfg["attn_hidden"], cfg["drop"]).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])

    def infer(loader):
        model.eval(); out = []
        with torch.no_grad():
            for e, m, l, r, _ in loader:
                lo, _, _ = model(e.to(dev), m.to(dev), l.to(dev), r.to(dev))
                out.append(torch.sigmoid(lo).cpu().numpy())
        return np.concatenate(out)

    best, bad, best_state = -1.0, 0, None
    for _ in range(cfg["epochs"]):
        model.train()
        for e, m, l, r, t in Ltr:
            opt.zero_grad()
            lo, _, _ = model(e.to(dev), m.to(dev), l.to(dev), r.to(dev))
            focal_bce(lo, t.to(dev), cfg["gamma"]).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        v = average_precision_score(yy[va], infer(Lva))
        if v > best:
            best, bad = v, 0
            best_state = {kk: vv.detach().clone() for kk, vv in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["patience"]: break
    model.load_state_dict(best_state)
    return (infer(Lte), te, model, tr) if keep_model else (infer(Lte), te)

def run_cv(E, MASK, labels=None, cfg=CFG, seed=0):
    oof = np.full(len(df), np.nan)
    for k in range(N_SPLITS):
        pred, te = train_fold(k, E, MASK, labels, cfg, seed); oof[te] = pred
    return oof
print("ready")

## 3.6 Seven seeds

One training run is noisy with this little data, so we train with 7 different seeds and average the
ranks. This is the slowest cell.

In [ ]:
seeds, per_seed = list(range(7)), []
t0 = time.time()
GREEN.start("network training, 7 seeds x 5 folds (GPU)")
for s in seeds:
    o = run_cv(E_mdna, M_mdna, seed=s); per_seed.append(o)
    print(f"  seed {s}: PR {per_fold(o)[1].mean():.3f}   ({time.time()-t0:.0f}s)")
GREEN.stop()
oof_dl = np.nanmean([rank_norm(o) for o in per_seed], 0)
print()
report(oof_dl, "deep model (7-seed ensemble)")

## 3.7 The blend

The final system is a 50/50 average of the network's ranks and the gradient boosting (ratios +
language) ranks. We fixed the 50/50 weight before looking at results. The other weights are only
printed to check the result doesn't depend much on it.

In [ ]:
blend = 0.5*rank_norm(oof_dl) + 0.5*rank_norm(oof_both)

print("all 763 rows:")
for nm, p in [("ratios only", oof_ratio), ("ratios + language", oof_both),
              ("deep model", oof_dl), ("BLEND", blend)]: report(p, nm)
print("\nstrict matched subset (609 rows):")
for nm, p in [("ratios only", oof_ratio), ("ratios + language", oof_both),
              ("deep model", oof_dl), ("BLEND", blend)]: report(p, nm, STRICT)

print("\nsensitivity to the blend weight (not tuned - shown for transparency):")
for w in [0, 0.25, 0.5, 0.75, 1.0]:
    print(f"  weight on deep model = {w:.2f}  ->  PR "
          f"{per_fold(w*rank_norm(oof_dl)+(1-w)*rank_norm(oof_both))[1].mean():.3f}")

## 3.8 Checks

Leak test: shuffle the labels and train again. If nothing leaks the answer, the model should drop to
chance (ROC-AUC around 0.5).

Then paired tests over the 5 folds and a bootstrap over matched pairs. With only 5 folds the smallest
possible Wilcoxon p-value is 0.0625, which means all 5 folds agreed.

In [ ]:
print("LEAK TEST - shuffled labels, all 5 folds")
y_shuffled = y[np.random.default_rng(0).permutation(len(y))]
sh = np.full(len(df), np.nan)
for k in range(N_SPLITS):
    pred, te = train_fold(k, E_mdna, M_mdna, labels=y_shuffled,
                          cfg=dict(CFG, epochs=15, patience=5)); sh[te] = pred
m = ~np.isnan(sh)
roc_sh = roc_auc_score(y_shuffled[m], sh[m])
print(f"  shuffled-label ROC-AUC {roc_sh:.3f}")
print("  PASS - nothing encodes the outcome" if roc_sh < 0.60 else "  *** FAIL - investigate ***")

In [ ]:
print("\nPAIRED TESTS across the five folds (PR-AUC)")
models = {"ratios only": oof_ratio, "ratios + language": oof_both,
          "deep model": oof_dl, "BLEND": blend}
for label, subset in [("all 763", None), ("strict 609", STRICT)]:
    print(f"\n  --- {label} ---")
    pf = {k: per_fold(v, subset)[1] for k, v in models.items()}
    for b in ["ratios only", "ratios + language", "deep model"]:
        d = pf["BLEND"] - pf[b]
        print(f"    BLEND vs {b:20s} {d.mean():+.3f}   t p={ttest_rel(pf['BLEND'],pf[b]).pvalue:.3f}"
              f"   wilcoxon p={wilcoxon(pf['BLEND'],pf[b]).pvalue:.3f}   wins {int((d>0).sum())}/5")

print("\nBOOTSTRAP over matched pairs (2,000 resamples), BLEND vs ratios + language")
A, B = rank_norm(blend), rank_norm(oof_both)
pairs = np.array(sorted(set(groups))); idx_of = {p: np.flatnonzero(groups==p) for p in pairs}
rng = np.random.default_rng(0); d_pr = []
for _ in range(2000):
    take = np.concatenate([idx_of[p] for p in rng.choice(pairs, len(pairs), replace=True)])
    if len(set(y[take])) < 2: continue
    d_pr.append(average_precision_score(y[take],A[take]) - average_precision_score(y[take],B[take]))
d_pr = np.array(d_pr)
print(f"  PR-AUC difference {average_precision_score(y,A)-average_precision_score(y,B):+.3f}"
      f"   95% CI [{np.percentile(d_pr,2.5):+.3f}, {np.percentile(d_pr,97.5):+.3f}]"
      f"   P(>0)={(d_pr>0).mean():.2f}")

## 3.9 What the network uses

Average gate weight of each branch, from one fold.

In [ ]:
pred, te, model, tr = train_fold(0, E_mdna, M_mdna, keep_model=True)
model.eval(); G, A_w = [], []
with torch.no_grad():
    for a in range(0, len(te), 64):
        i = te[a:a+64]
        ns = CFG["n_sents"]
        _, g, aw = model(torch.from_numpy(np.ascontiguousarray(E_mdna[i, :ns]).astype(np.float32)).to(dev),
                         torch.from_numpy(np.ascontiguousarray(M_mdna[i, :ns])).to(dev),
                         torch.from_numpy(prep(tr,i,Xl_all)).to(dev),
                         torch.from_numpy(prep(tr,i,Xr_all)).to(dev))
        G.append(g.cpu().numpy()); A_w.append(aw.cpu().numpy())
G, A_w = np.concatenate(G), np.concatenate(A_w)

print("average gate weight per input block:")
for nm, v in zip(["text (FinBERT)", "language features", "accounting ratios"], G.mean(0)):
    print(f"  {nm:22s} {v:.3f}")
plt.figure(figsize=(4.5,3))
plt.bar(["text","language\nfeatures","ratios"], G.mean(0))
plt.ylabel("average gate weight"); plt.title("What the network relied on"); plt.tight_layout(); plt.show()

In our final run the gate gave text 0.23, language features 0.49 and ratios 0.27, and taking out the
FinBERT branch didn't make the network worse (0.789 without it, 0.783 with it). So most of the text
signal is coming from the engineered features, mainly the auditor's CARO flags, and not from
FinBERT. With ~490 training reports per fold that isn't too surprising.

## 3.10 Hypotheses

H3: does the warning show up early? Same predictions, scored separately 1, 2 and 3 years before.
H4: which section carries the signal? The text-only network trained on the MD&A, on the auditor's
report and on both. Last cell: companies whose accounts still look healthy (Altman Z above median).

In [ ]:
print("H3 - does the warning appear early, or only just before the event?")
print(f"  {'horizon':28s} {'n':>5s} {'ratios only':>12s} {'full system':>12s}")
for h in ["t-1","t-2","t-3"]:
    sub = (df.horizon == h).to_numpy()
    if sub.sum() < 20: continue
    print(f"  {h:28s} {int(sub.sum()):5d} {per_fold(oof_ratio,sub)[1].mean():12.3f}"
          f" {per_fold(blend,sub)[1].mean():12.3f}")
print("  -> a flat profile means the signal survives three years before admission")

In [ ]:
print("\nH4 - is the signal only the auditor's flags?")
# up to 96 MD&A sentences followed DIRECTLY by up to 96 auditor sentences: the packed GRU reads
# the first `length` slots, so the real sentences must be contiguous
L_m = np.minimum(M_mdna.sum(1), 96); L_a = np.minimum(M_aud.sum(1), 96)
E_both = np.zeros_like(E_mdna)
for i in range(len(df)):
    E_both[i, :L_m[i]] = E_mdna[i, :L_m[i]]
    E_both[i, L_m[i]:L_m[i] + L_a[i]] = E_aud[i, :L_a[i]]
M_both = np.arange(MAX_SENTS)[None, :] < (L_m + L_a)[:, None]
for nm, EE, MM, ns in [("management discussion", E_mdna, M_mdna,  96),
                       ("auditor's report",      E_aud,  M_aud,   96),
                       ("both sections",         E_both, M_both, 192)]:
    o = run_cv(EE, MM, cfg=dict(CFG, n_sents=ns))
    report(o, nm)
print("  -> final run: the auditor's report is the stronger text (text-only PR-AUC 0.679 against 0.547)")

In [ ]:
print("\nWhere language matters most: companies whose accounts look healthy")
z = pd.to_numeric(df["altman_z_em"], errors="coerce")
safe = ((z > z.median()) & z.notna()).to_numpy()
print(f"  {int(safe.sum())} company-years with above-median Altman Z, "
      f"{int(y[safe].sum())} of them distressed (base rate {y[safe].mean():.2f})")
for nm, p in [("ratios only",oof_ratio),("ratios + language",oof_both),("BLEND",blend)]:
    report(p, nm, safe)
print("  -> compare these gaps with the full-sample gaps in Section 7")

## 3.11 Final numbers in our report

The numbers in the report come from the same pipeline run as scripts on the lab GPU, with every
prediction saved (`final_run/` in the repo). This notebook should give numbers close to these; small
differences come from GPU randomness.

| Model | PR-AUC (763 rows) | PR-AUC (strict 609) |
| --- | --- | --- |
| Altman Z'' | 0.712 | 0.778 |
| Gradient boosting, ratios | 0.747 | 0.786 |
| Gradient boosting, ratios + language | 0.795 | 0.822 |
| Network, 7 seeds | 0.783 | 0.823 |
| Blend (final system) | 0.807 | 0.838 |

H1 language alone separates the groups: yes. H2 language adds to the ratios: yes, in all 5 folds.
H3 the warning is early: yes (0.842, 0.814, 0.823 at 1, 2, 3 years). H4: the auditor's report is
the stronger text, but the MD&A also adds something.

# Part 4: Green AI

## 4.1 Energy used by this run

Time and GPU energy of each heavy step, with CO2 from the India grid factor. Training the networks is
by far the biggest part. If the FinBERT vectors were already cached, that step costs almost nothing.

In [ ]:
bill = GREEN.table()
display(bill)
gpu_wh = bill.gpu_wh.fillna(0).sum(); g = bill.g_co2e_gpu.fillna(0).sum()
print(f"total: {bill.seconds.sum()/60:.0f} min, {gpu_wh:.2f} Wh of GPU energy, {g:.2f} g CO2e (GPU, India grid)")

## 4.2 The greener options

Using the predictions already made above, we check what we lose if we train fewer network seeds, or
drop the network and keep only gradient boosting on ratios + language features. No extra training:
the energy of each option is worked out from the measured steps.

In our full study (report Section 10, 82 configurations), gradient boosting alone scored 0.795 against
0.807 for about 42 times less GPU energy, and 3 seeds instead of 7 scored 0.805 for about half the energy.

In [ ]:
# the greener options, worked out from this run's own predictions (no extra training)
def cost(step, col):
    r = bill[bill.step.str.startswith(step)]
    return float(r[col].fillna(0).sum()) if len(r) else 0.0

col  = "gpu_wh" if bill.gpu_wh.fillna(0).sum() > 0 else "seconds"   # seconds if codecarbon didn't work
unit = "GPU Wh" if col == "gpu_wh" else "seconds"
emb, net, gb = cost("FinBERT", col), cost("network training", col), cost("gradient boosting", col)
per_seed_cost = net / len(per_seed)

opts = []
for n in (7, 5, 3, 1):
    p = 0.5*rank_norm(np.nanmean([rank_norm(o) for o in per_seed[:n]], 0)) + 0.5*rank_norm(oof_both)
    opts.append((f"blend, network with {n} seed{'s' if n > 1 else ''}", per_fold(p)[1].mean(), emb + n*per_seed_cost + gb))
opts.append(("gradient boosting only (no network)", per_fold(oof_both)[1].mean(), gb))

green = pd.DataFrame(opts, columns=["option", "PR-AUC", unit])
green["saving vs 7 seeds"] = [f"{100*(1 - c/green[unit].iloc[0]):.0f}%" for c in green[unit]]
display(green.round(3))

plt.figure(figsize=(6.5, 3.8))
plt.scatter(green[unit], green["PR-AUC"])
for _, r in green.iterrows():
    plt.annotate(r.option, (r[unit], r["PR-AUC"]), textcoords="offset points", xytext=(6, 4), fontsize=8)
plt.xlabel(unit + " (this run)"); plt.ylabel("PR-AUC"); plt.title("Accuracy against energy")
plt.tight_layout(); plt.show()

# Part 5: Test it on a new annual report

`score.py` (in `final_run/` of our repo) takes an annual report PDF, finds the sections, computes the
same features and ratios, and scores it with our final models (trained on all 763 company-years).
It prints a risk score out of 100 with the reasons and saves an HTML score card.

Run all uses the sample report (SEL Manufacturing FY2017). To test your own report, set
`USE_SAMPLE = False` in the next cell and run it: a button appears to upload the PDF from your
computer. It has to be the full annual report, not just the MD&A. Then run the two cells after it.
Set `USE_NETWORK = False` to use only gradient boosting (the green option).

In [ ]:
USE_SAMPLE = True      # False = upload your own annual report PDF when you run this cell
FY = None              # the report's fiscal year, e.g. 2024 (None = read it from the report)

if USE_SAMPLE:
    PDF = DATA/"score"/"reports"/"SEL_Manufacturing"/"FY2017.pdf"
else:
    from google.colab import files
    up = files.upload()                                    # pick the PDF
    name = next(iter(up))
    folder = Path("/content/uploaded"); folder.mkdir(exist_ok=True)
    PDF = folder/name
    PDF.write_bytes(up[name])
print("report:", PDF)

In [ ]:
USE_NETWORK = True
SCORE = DATA/"score"
os.environ["BPP_LM_DICTIONARY"] = str(SCORE/"loughran_mcdonald.csv")
card_dir = Path("/content/score_out"); card_dir.mkdir(exist_ok=True)
cmd = [sys.executable, "/content/bpp-repo/final_run/score.py", str(PDF),
       "--models", str(SCORE/"final_models"), "--out", str(card_dir)]
if FY:
    cmd += ["--fy", str(FY)]
if not USE_NETWORK:
    cmd.append("--no-dl")

GREEN.start("scoring one report")
res = subprocess.run(cmd, capture_output=True, text=True)
GREEN.stop()
print(res.stdout[-6000:])
if res.returncode != 0:
    print(res.stderr[-3000:])

In [ ]:
# the score card
from IPython.display import HTML
card = max(card_dir.glob("*_risk.html"), key=os.path.getmtime)
HTML(card.read_text())